In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
from scipy.stats import pearsonr
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline

pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 20)

atlas_samples_fn ="/well/ludwig/users/cfo155/tissueMap/methcalls/meth_around_gene/MANE.GRCh38.v1.0.refseq_genomic.gene.flank20000.nwin20.all_meth.closest.1000.txt"
tissue_specific_fn = "/well/ludwig/users/qwb368/public_data/tissue_specific_genes_HPA_Magda/tissue_specific_genes.tsv"
solid_gene_expr_fn = "/well/ludwig/users/qwb368/public_data/HPA_GTEx_consensus/rna_tissue_consensus_formatted.tsv"
immune_gene_expr_fn = "/well/ludwig/users/qwb368/public_data/HPA_immune/rna_immune_cell_formatted.tsv"

atlas_samples_df = pd.read_csv(atlas_samples_fn,sep="\t")

tissue_specific_df = pd.read_csv(tissue_specific_fn, sep='\t')
tissue_specific_df = tissue_specific_df.set_index('Ensembl')
tissue_specific_genes_df = tissue_specific_df[tissue_specific_df["specificity"].isin(["Tissue enhanced", "Group enriched", "Tissue enriched"])]

solid_gene_expr_df = pd.read_csv(solid_gene_expr_fn,sep="\t")
solid_gene_expr_df = solid_gene_expr_df.set_index('Name')
solid_gene_expr_df.index.names = ['Ensembl']
immune_gene_expr_df = pd.read_csv(immune_gene_expr_fn,sep="\t")
immune_gene_expr_df = immune_gene_expr_df.set_index('Name')
immune_gene_expr_df.index.names = ['Ensembl']
normal_tissue_gene_expr_df = solid_gene_expr_df.merge(immune_gene_expr_df,on='Ensembl')
normal_tissue_gene_expr_df = normal_tissue_gene_expr_df.drop(columns=['Description_y'])
normal_tissue_gene_expr_df = normal_tissue_gene_expr_df.rename(columns={'Description_x': 'gene'})

print(atlas_samples_df)
print(tissue_specific_genes_df)
print(normal_tissue_gene_expr_df)

In [ ]:
atlas_samples_df = atlas_samples_df[~atlas_samples_df['chr'].isin(['chrX', 'chrY'])]
print(atlas_samples_df)

In [ ]:
meta_cols = ['chr', 'start', 'end', 'gene', 'feature', 'idx']

sample_cols = [c for c in atlas_samples_df.columns if c not in meta_cols]

parsed = (
    pd.DataFrame(sample_cols, columns=["col"])
      .assign(
          tissue=lambda df: df["col"].str.split("_").str[-2],
          mod=lambda df: df["col"].str.split("_").str[-1]
      )
      .assign(tissue_mod=lambda df: df["tissue"] + "_" + df["mod"])
)

col_to_group = parsed.set_index("col")["tissue_mod"].to_dict()

grouped_values = (
    atlas_samples_df[sample_cols]
      .groupby(col_to_group, axis=1)
      .mean()
)

result_df = pd.concat([atlas_samples_df[meta_cols], grouped_values], axis=1)

print(result_df)

In [ ]:
result_df = result_df.drop(result_df.filter(regex="Brain").columns, axis=1)
print(result_df)

In [ ]:
meta_cols = ['chr', 'start', 'end', 'gene', 'feature', 'idx']

value_cols = [c for c in result_df.columns if c not in meta_cols]

long = result_df.melt(
    id_vars=meta_cols,
    value_vars=value_cols,
    var_name='tissue_mod',
    value_name='value'
)

long[['tissue', 'mod']] = long['tissue_mod'].str.rsplit('_', n=1, expand=True)

result_long = (
    long
    .pivot_table(
        index=meta_cols + ['tissue'],
        columns='mod',
        values='value',
        aggfunc='first'
    )
    .reset_index()
)

result_long.columns.name = None

print(result_long)

In [ ]:
result_long = result_long.drop(columns=['chr', 'start', 'end', 'feature'])

long = result_long.melt(
    id_vars=['gene', 'tissue', 'idx'],
    value_vars=['hmC', 'mC', 'umC'],
    var_name='mod',
    value_name='value'
)

wide = long.pivot_table(
    index=['gene', 'tissue'],
    columns=['mod', 'idx'],
    values='value',
    aggfunc='first'
)

wide = wide.sort_index(axis=1, level=[0, 1])

wide.columns = [f'{mod}_{idx}' for mod, idx in wide.columns]

final_atlas_df = wide.reset_index()

print(final_atlas_df)

In [ ]:
tissue_specific_df_deduplicated = tissue_specific_genes_df[~tissue_specific_genes_df.index.duplicated(keep='first')]
print(tissue_specific_df_deduplicated)

In [ ]:
normal_tissue_specific_df = normal_tissue_gene_expr_df.join(tissue_specific_df_deduplicated, how='inner')
normal_tissue_specific_df = normal_tissue_specific_df.drop(columns=['Gene'])
print(normal_tissue_specific_df)

In [ ]:
normal_tissue_specific_df = normal_tissue_specific_df.reset_index()
normal_tissue_specific_df = normal_tissue_specific_df.set_index('gene')
normal_tissue_specific_df = normal_tissue_specific_df.drop(columns=['Ensembl', 'specificity', 'tissue'])
print(normal_tissue_specific_df)

In [ ]:
normal_tissue_specific_df.columns = normal_tissue_specific_df.columns.str.title()
normal_tissue_specific_df.rename(columns={'Heart Muscle': 'Heart', 'Memory Cd4 T-Cell': 'CD4-T-cells', 'Neutrophil':'Neutrophils', 'Naive B-Cell':'B-cells', 'Nk-Cell':'NK-cells', 'Cerebral Cortex':'Brain', 'Memory Cd8 T-Cell':'CD8-T-cells', 'Eosinophil':'Eosinophils', 'Classical Monocyte':'Monocytes'}, inplace=True)
print(normal_tissue_specific_df)

In [ ]:
expr_long = (
    normal_tissue_specific_df
    .reset_index()
    .melt(
        id_vars='gene',
        var_name='tissue',
        value_name='expression'
    )
)

total_df = final_atlas_df.merge(
    expr_long,
    on=['gene', 'tissue'],
    how='inner'
)

print(total_df)

In [ ]:
def impute_nearest_in_row(X):
    """
    For each NaN in X[i, j], replace it with the value of the closest
    non-NaN element in the same row i (by column index distance).
    If a row is all NaNs, it is left unchanged.
    """
    X = X.copy().astype(float)
    n_rows, n_cols = X.shape

    for i in range(n_rows):
        row = X[i]
        not_nan_idx = np.where(~np.isnan(row))[0]

        if not_nan_idx.size == 0:
            continue

        nan_idx = np.where(np.isnan(row))[0]

        for j in nan_idx:
            nearest_col = not_nan_idx[np.argmin(np.abs(not_nan_idx - j))]
            row[j] = row[nearest_col]

    return X


TARGET_COL = "expression"
GROUP_COL = "tissue"
NON_FEATURE_COLS = ["gene", "tissue", TARGET_COL]

feature_cols = [c for c in total_df.columns if c not in NON_FEATURE_COLS]
print(feature_cols)

X_nan = total_df[feature_cols].to_numpy(dtype=float)
y_tpm = total_df[TARGET_COL].to_numpy(dtype=float)
tissues = total_df[GROUP_COL].astype(str).to_numpy()

X_all = impute_nearest_in_row(X_nan)
y_all = np.log(np.clip(y_tpm, 1e-2, None))

# ----------------------------
# MODEL: deep neural net
# ----------------------------

model = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        ("mlp", MLPRegressor(
            hidden_layer_sizes=(256, 128, 64),
            activation="relu",
            solver="adam",
            alpha=1e-4,
            learning_rate="adaptive",
            batch_size=256,
            learning_rate_init=1e-3,
            max_iter=200,
            random_state=42,
            early_stopping=True,
            validation_fraction=0.1,
            n_iter_no_change=20,
            tol=1e-4,
            verbose=True
        )),
    ]
)

#model = Pipeline(
    #steps=[
        #("scaler", StandardScaler()),
        #("rf", RandomForestRegressor(
            #n_estimators=300,
            #max_depth=30,
            #min_samples_split=5,
            #min_samples_leaf=2,
            #max_features="sqrt",
            #bootstrap=True,
            #random_state=42,
            #verbose=2,
        #)),
    #]
#)

# ----------------------------
# LEAVE-ONE-TISSUE-OUT CV
# ----------------------------

results = []
unique_tissues = pd.unique(total_df[GROUP_COL].astype(str))

for held_out in unique_tissues:
    test_mask = (tissues == held_out)
    train_mask = ~test_mask

    X_train, y_train = X_all[train_mask], y_all[train_mask]
    X_test, y_test   = X_all[test_mask], y_all[test_mask]

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    if np.std(y_test) == 0 or np.std(y_pred) == 0:
        r = np.nan
        r_p = np.nan
    else:
        r, r_p = pearsonr(y_test, y_pred)

    fold_res = {
        "held_out_tissue": held_out,
        "n_train": int(train_mask.sum()),
        "n_test": int(test_mask.sum()),
        "mse": float(mse),
        "r2": float(r2),
        "pearson_r": float(r) if np.isfinite(r) else np.nan,
    }
    results.append(fold_res)

    print(
        f"[Hold out: {held_out}] "
        f"n_test={fold_res['n_test']} | "
        f"MSE={fold_res['mse']:.6g} | "
        f"R2={fold_res['r2']:.4f} | "
        f"Pearson r={fold_res['pearson_r']:.4f}"
    )

In [ ]:
results_df = pd.DataFrame(results)
print(results_df)
print(results_df[["mse", "r2", "pearson_r"]].mean())

results_df.to_csv("combined_results.csv", index=False)